# L7a: what Spark reads: scan tasks, row groups and rows

This is part A of L7's demos. Steps 1-6 show how Spark reads a folder of Parquet files: which folders it keeps, how
it cuts files into byte ranges, what each footer holds, which row groups it skips, and how many tasks and rows each
query ends up with. Exercise 1 shows when `show` needs every task. Exercises 2-6 are practice on the same data, to try
on your own.

End with `%stop_session` to stop Glue billing you!

To show you what happens inside, we are going to use a small vibecoded `sparkmeter` (`sm`) library. If you see `sm.something` this is NOT a Spark operation, this is a wrapper usually handling getting spark performance data, doing visualizations or analyzing query statistics. Don't over-rely on it, this is to skip some details for now (and because Glue notebooks don't show the Spark UI in the browser, though it can be turned on with event logs and a Spark history server, which we skip here)

Commands without sm are either Spark/PySpark commands, or AWS Glue magic.

AWS Glue setup config. We specify 3 workers (Spark driver and 2 Spark executors) and stop the cluster after a 15-minute idle timeout to limit charges if you forget to stop session.

But: **DON'T FORGET TO STOP THE SESSION**

In [ ]:
%glue_version 5.1
%worker_type G.1X
%number_of_workers 3
%idle_timeout 15

We also ask Glue to load our sparkmeter script from S3, as Glue doesn't have bootstrap scripts (in contrast to EMR).

In [ ]:
%%configure
{"--extra-py-files": "s3://436c-2026w1/l7/student/sparkmeter.py"}

**Step 0.** Run the cell below: it starts the Glue session (30 to 60 s the first time) and loads `sparkmeter`. It prints the Spark version and the number of cores.

In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import functions as F
sc = SparkContext.getOrCreate()
spark = GlueContext(sc).spark_session
import sparkmeter as sm
sm.setup(spark)      # to keep your runs after the session: sm.setup(spark, save_to="s3://436c-glue-<your account>/sparkmeter/")

## Part 1: reading files

**Words used in part 1**

- `PartitionFilters` are conditions on the column Spark makes from the folder names, such as `pickup_month`. Spark
  checks them against the folder names and drops the folders that can't match, before any scan task starts. A join
  can also create such a filter while the query runs: **dynamic pruning** (exercise 6).
- `PushedFilters` are conditions Spark gives to the Parquet reader. The reader checks them against each row group's
  footer statistics (min, max, null count) and skips the row groups that can't match. A condition listed there may
  still skip nothing: compare the rows read with the row-group sizes to see what was skipped.
- A **scan task** gets one byte range of a large file, or a bundle of pieces of small files. A **reading task** is a
  scan task whose input holds at least one row group to read. The other scan tasks finish with 0 rows.
- A **Parquet partition** is a folder, such as `pickup_month=2012-06/`.
  A **Spark data partition** is the input of one scan task. An **`Exchange` output partition** receives the rows that
  an `Exchange` sends to it, all with the same key.

![Reading a folder of Parquet files, in six steps](https://raw.githubusercontent.com/cpsc436c-2026w1/l7-spark-glue/main/img/l7-reading-steps.svg)

**Step 1. The data and how to organize it: 36 months of taxi trips**

Data: 520,062,606 trips (2011 to 2013), in one folder per month: `pickup_month=2011-01/`, and so on.

We deal with two data representations (two copies of data organized differently)

- `month_folders` (8.7 GB): one file per month with **one row group**; pickup time stored as local time without specifying time zone, as the city publishes it. Used in steps 1-5 and exercises 1, 2, 5 and 6. This is the original data we used in L5-L6.
- `month_folders_sorted` (6.9 GB): also one file per month. Each file is sorted by pickup time and has 11 or 12
  **row groups** of about 1.3 million trips each. The pickup time is marked as UTC: step 2 shows `tz=UTC`. Spark 3.5
  checks a time condition against a row group's min and max only when the time is marked as UTC. Spark doesn't check
  the values themselves, so we kept the city's local clock times and only added the UTC mark: 08:00 stays 08:00. Used in steps 2 and 6 and exercises 2-4.

`sm.tree` previews the data in S3. 

**Check:** how many Parquet partitions (folders) in each version, and how many row groups?

In [ ]:
month_folders = "s3://436c-2026w1/l7/layouts/by-month/"
month_folders_sorted = "s3://436c-2026w1/l7/layouts/by-month-utc-sorted/"

sm.tree(month_folders)
print()
sm.tree(month_folders_sorted, row_groups=True)

**What to look at.** The same folder names in both copies, one file in each folder. The sorted copy's tree adds each
file's row groups, read from its footer: 11 for the months shown. The sorted files are 175-187 MB, so each row group
is about 16 MB as stored (compressed), about 22 MB once decompressed.

The cell below creates a DataFrame for `month_folders`. Creating it isn't entirely lazy: Spark lists the files, turns
the folder names into a column, and reads one data file's footer to learn the columns, but it reads no rows.
`printSchema()` shows the columns Spark already learned; it starts no job. `show(3)` is the action that reads rows.

**Predict:** where in the schema is `pickup_month`, and what type is it?

In [ ]:
trips_by_month = spark.read.parquet(month_folders)
trips_by_month.printSchema()
trips_by_month.select("pickup_month", "tpep_pickup_datetime", "total_amount").show(3)

**What to look at.** The last schema line: `pickup_month` is a `string` that Spark made from the folder names; the
files hold 17 columns, the DataFrame 18. `tpep_pickup_datetime` is a `timestamp_ntz`: a time without a time zone. The
3 rows come from whichever file was read first (August 2012 in our run).

How `show(3)` ran: one job with **one** scan task, not 72. This query has no condition and no sort, so the first byte
range Spark tried (August 2012's file in our run) already had enough rows, and Spark stopped after that task. The task
read 4,096 rows because Spark's Parquet reader reads rows in batches of 4,096 by default
(`spark.sql.parquet.columnarReaderBatchSize`). With a sort before `show`, as in `orderBy(...).show(3)`, all 72 tasks
must run, because the top rows could be in any file.

**How `spark.read` turns files into tasks** (the diagram at the top of part 1; you'll see these numbers in every step)

- `spark.read.parquet(month_folders)` runs at once, not lazily: Spark lists the files in S3, turns the
  `pickup_month=...` folder names into a column, and reads one data file's footer for the schema. It reads no rows.
- Spark plans the byte ranges when an action runs, or earlier if you call `getNumPartitions()`. The driver works out
  a range size from the file sizes in the listing alone:
  range size = min(128 MiB, max(4 MiB, (bytes of all files + 4 MiB × number of files) ÷ 8)).
  128 MiB is the largest range by default (`spark.sql.files.maxPartitionBytes`). 4 MiB is the cost Spark counts for
  opening each file (`spark.sql.files.openCostInBytes`). On Glue with 2 executors of 4 cores each, Spark's default
  parallelism is 8: 8 tasks can run at once.
- Spark cuts large files into ranges of that size. It packs small pieces into one task until the next piece would
  pass that size, counting 4 MiB extra for each piece. Each range, or bundle of pieces, is one **scan task**.
- Why byte ranges and not row groups: the S3 listing already gives the driver each file's size, so it can plan ranges
  (a start and a length) without opening any file. Where a row group starts and how long it is is written only in
  its file's footer. Planning by row groups would mean reading every footer on the driver before any task could
  start. So Spark plans byte ranges first, and each task reads the footers it needs.
- For each piece it holds, a scan task reads that file's footer and keeps the row groups whose middle falls inside
  the piece's range. A scan task that keeps one is a **reading task**. The others finish with 0 rows.
- Iceberg and Delta are table formats: they keep extra metadata outside the Parquet files. Iceberg can store each
  file's row-group offsets (`split_offsets`) and then plan splits at row-group boundaries, aiming for 128 MB by
  default (`read.split.target-size`). Delta's table log can store each file's row count and the min and max of its
  first 32 columns by default, and Delta skips whole files with them. Spark then cuts the files it keeps into byte
  ranges as above.

**Step 1b. A rare filter: how much does `show` read?**

`show(3)` above read one batch of 4,096 rows. Here very few trips pass the filter, so the task may read many batches
before it finds enough rows. Queries that make `show` run every task (a group-by, a count per folder) are at the end:
"When `show` needs every task".

**Predict:** how many scan tasks run, and how many rows are read?

In [ ]:
months_by_time = spark.read.parquet(month_folders).select("pickup_month", "tpep_pickup_datetime", "total_amount")
rare_trips = months_by_time.where("total_amount > 500")
rare_trips.show(3)

The query `show(3)` runs is `rare_trips` with a limit of 4 rows (3 to print, one more to know whether there are more).
`explain()` prints the plan Spark will run; read it from the bottom (the scan) up.

In [ ]:
rare_trips.limit(4).explain()

In [ ]:
sm.capture("show(3), rare filter")
sm.stage_table("show(3), rare filter")

**What to look at.** Read the plan from the bottom. `FileScan` shows `PushedFilters: [..., GreaterThan(total_amount,500.0)]`.
Above it, `Filter` applies the condition and `Project` keeps the three columns. `CollectLimit 4` is at the top.

There is no `Exchange`: the scan task applies the limit itself and stops once it has 4 rows. The stage table shows
1 scan task, but 5,189,632 rows read. Trips over $500 are rare, and the file's one row group has amounts up to $600,
so its footer can't rule it out. The task reads batch after batch until it finds 4.

**Step 2. What a Parquet file's footer holds**

Every Parquet file ends with a footer. For each row group and each column, the footer records the size. It may also
record statistics such as min, max and null count, if the writer stored them. Each scan task reads the footer before
the data, so that it can skip row groups that can't match.

`sm.footer` reads the footer of June's file (a few KB) in each copy and shows it to you.

In [ ]:
sm.footer(month_folders + "pickup_month=2012-06/part-0.parquet")

In [ ]:
sm.footer(month_folders_sorted + "pickup_month=2012-06/part-0.parquet")

**Predict:** In `month_folders_sorted` (file with sorted rows and multiple row groups), how many row groups could contain trips on 14 June?

In [ ]:
sm.footer(month_folders_sorted + "pickup_month=2012-06/part-0.parquet",
          column="tpep_pickup_datetime", between=("2012-06-14", "2012-06-15"))

**What to look at.** In `month_folders`, June has one row group whose pickup times cover the whole month, so a
condition on 14 June can't rule it out. In `month_folders_sorted`, June has 12 row groups stored as
`timestamp[us, tz=UTC]`. Most cover about 2.5 days, and the last about 1.5 days.

In the last table, only **one** row group could match: the sixth, from 13 June 21:15 to 16 June 10:07, the only row
with `could match` = `True`. A day can span two row groups, but 14 June falls inside this one.

**Step 3. June data queried by pickup time, on the original 36 Parquet monthly partitions, one row group each.**

Spark can skip a Parquet partition (a folder) **only when the condition is on the folder column**, `pickup_month`. Here the condition is on
`tpep_pickup_datetime`, a column inside the files.

**Predict:** how many of the 36 folders does Spark read?

In [ ]:
june_by_time = (spark.read.parquet(month_folders)
                .where("tpep_pickup_datetime >= '2012-06-01' AND tpep_pickup_datetime < '2012-07-01'")
                .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
june_by_time.explain()
june_by_time.collect()

In [ ]:
sm.capture("june by time", june_by_time)
sm.checklist("june by time")

**What to look at.** `collect()` brought the result to the driver: one row, the count and the total. In the plan,
`PartitionFilters: []` is empty: no folder can be dropped. The checklist shows all
36 folders kept, 72 scan tasks, 36 of them reading, and 520,062,606 rows read to count 15,096,468 trips in June.
Spark doesn't know that `pickup_month` was made from the pickup time.

**Step 4. The fix: add a condition on the folder column**

The code adds `pickup_month = '2012-06'` to step 3's query, then runs `explain()`, `collect()` and the checklist.
The answer can't change: every June trip is in June's folder.

**Predict:** how many folders, scan tasks and rows now?

In [ ]:
june_by_folder = (spark.read.parquet(month_folders)
                  .where("pickup_month = '2012-06'")
                  .where("tpep_pickup_datetime >= '2012-06-01' AND tpep_pickup_datetime < '2012-07-01'")
                  .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
june_by_folder.explain()
june_by_folder.collect()

In [ ]:
sm.capture("june by folder", june_by_folder)
sm.checklist("june by folder")

**What to look at.** `PartitionFilters` now holds `pickup_month = 2012-06`: 1 folder kept, before any task starts.
June's one file is cut into 8 byte ranges (8 scan tasks). The line `1 of 8 scan tasks still hold a row group` says
only one reads. The file has one row group, and it belongs to the range that holds its middle (see "How
`spark.read` turns files into tasks"). 15,096,468 rows read, the same 15,096,468 trips.

**Step 5. One day, 14 June, on `month_folders`**

The folder condition keeps June's file. Inside it, `PushedFilters` sends the time condition to each footer. But June
has one row group spanning the whole month (step 2), so the footer can't rule it out.

**Predict:** how many rows does Spark read to count one day's trips?

In [ ]:
one_day = "pickup_month = '2012-06' AND tpep_pickup_datetime >= '2012-06-14' AND tpep_pickup_datetime < '2012-06-15'"

day_one_row_group = (spark.read.parquet(month_folders).where(one_day)
                     .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
day_one_row_group.collect()

In [ ]:
sm.capture("one day, 1 row group", day_one_row_group)
sm.checklist("one day, 1 row group")

**What to look at.** `PushedFilters` lists the conditions for 14 June. The footer can't rule out June's only row
group, so Spark reads all 15,096,468 rows to find the 528,608 trips of that day. Spark 3.5 also doesn't compare
local-time timestamps with footer statistics.

**Step 6. The fix: the same query on `month_folders_sorted`**

The code runs step 5's condition on the sorted copy. June's file has 12 row groups of about 2.5 days each. The pickup
times are marked as UTC (step 1's data card), so Spark can use each row group's min and max to rule it out.

**Predict:** how many rows now?

In [ ]:
day_sorted = (spark.read.parquet(month_folders_sorted).where(one_day)
              .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
day_sorted.collect()

In [ ]:
sm.capture("one day, sorted", day_sorted)
sm.checklist("one day, sorted")

**What to look at.** The line `6. read` shows 1,310,100 rows read for the same 528,608 trips. Those are the rows of
one row group, the sixth in step 2's footer. Spark skipped the other 11 June row groups using only their footer
statistics, without reading their data.

**Reading files: rules of thumb**

- **Filter on the folder column** in every query on folders, and join it to the data condition with `AND`. If `OR`
  joins a folder condition to a condition on a data column, Spark keeps every folder, because matching rows can be in
  any of them.
- **Sort before writing** by the column you filter on, with several row groups per file.
- **Mark times as UTC** if time conditions should skip row groups. Spark 3.5 doesn't use the footer min and max for
  local times (`timestamp_ntz`, a time with no time zone). It also doesn't use them for the older time format that
  Spark itself writes by default. Before Spark writes the column, set `spark.sql.parquet.outputTimestampType` to
  `TIMESTAMP_MICROS`.
- **Compare a data column itself** with a value. A function of it usually hides it from the footer; Spark rewrites a
  few, such as `to_date`. Functions of the folder column still drop folders.
- **Check the plan and the stage table**: `PartitionFilters` and `PushedFilters` say what Spark will try, and the
  rows read say what it skipped.

## When `show` needs every task

**Exercise 1. When does `show(3)` need every task?**

In step 1, `show(3)` ran one scan task. Here, a group-by per payment type, then a count per month (the folder
column).

**Predict:** for the group-by, how many of the 72 scan tasks run, and how many rows are read? Why can't `show(3)`
stop after one task here?

In [ ]:
months_by_time_and_type = spark.read.parquet(month_folders).select("payment_type", "total_amount")
months_by_time_and_type.groupBy("payment_type").count().show(3)

In [ ]:
sm.capture("show(3), group-by")
sm.stage_table("show(3), group-by")

In [ ]:
counts_per_month = trips_by_month.groupBy("pickup_month").count()
counts_per_month.show(3)

Where does the plan say how the counts were made? `show(3)` above ran the count with a limit of 4 rows added, as a
query of its own. `counts_per_month` itself hasn't run yet, so `explain()` alone would print its plan from before
running.
The cell below runs it first with `collect()` (36 rows, one per month), then prints its final plan.

In [ ]:
counts_per_month.collect()
counts_per_month.explain()

In [ ]:
sm.capture("count per folder", counts_per_month)
sm.checklist("count per folder")

In [ ]:
# Two more, not run here (each reads all 520 M rows or holds millions of rows on the driver). Our run:
#
# A sort: every scan task runs (72, 8 at a time), each keeps its own top 4 (show(3) asks for 4 rows), the driver merges them.
# trips_by_month.orderBy(F.desc("total_amount")).show(3)
#
# More rows than one task holds: one task reads a row group of 2.15 M rows, too few, so Spark runs a second
# one-task job for the rest.
# spark.read.parquet("s3://436c-2026w1/l4/june-123-row-groups/").head(3_000_000)

**What to look at.** The group-by runs all 72 scan tasks and reads every row. The counts then go through an
`Exchange`, and only after that does `show` take its rows.

The count per `pickup_month` also runs all 72 scan tasks, but the checklist's `6. read` line says 36 rows: one
row-count record from each file's footer, not 36 trips. Each file is one month, so the footers give the monthly counts
without reading the data. The plan says so: in `== Final Plan ==`, the `FileScan` line has
`EnablePushdownAggregate: true`, Glue's name for this. The `== Initial Plan ==` below it, from before the run, doesn't.
A query with a limit stops early only when nothing before the limit needs all the rows.

## Practice: exercises 2-6, on your own

**Exercise 2. The first weeks of three summer months**

`summer` keeps three folders; `first_weeks` is an `OR` of three time ranges, which Spark also sends to the footers.
The same query runs on both copies.

**Predict:** for each copy, how many rows are read?

In [ ]:
summer = "pickup_month IN ('2012-06', '2012-07', '2012-08')"
first_weeks = ("(tpep_pickup_datetime >= '2012-06-01' AND tpep_pickup_datetime < '2012-06-08') OR "
               "(tpep_pickup_datetime >= '2012-07-01' AND tpep_pickup_datetime < '2012-07-08') OR "
               "(tpep_pickup_datetime >= '2012-08-01' AND tpep_pickup_datetime < '2012-08-08')")

weeks_one_row_group = (spark.read.parquet(month_folders).where(summer).where(first_weeks)
                       .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
weeks_one_row_group.collect()

In [ ]:
sm.capture("first weeks, 1 row group", weeks_one_row_group)
sm.checklist("first weeks, 1 row group")

In [ ]:
weeks_sorted = (spark.read.parquet(month_folders_sorted).where(summer).where(first_weeks)
                .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
weeks_sorted.collect()

In [ ]:
sm.capture("first weeks, sorted", weeks_sorted)
sm.checklist("first weeks, sorted")

**What to look at.** Both checklists: 3 folders kept, 9 scan tasks, the same 9,839,601 trips. `month_folders`
reads all 43,857,527 rows of the three months. `month_folders_sorted` reads 11,610,900, about 3 row groups of each
month (at about 1.3 million rows each, see the data card): the ones whose min and max overlap a first week.

**Exercise 3. The same three months, trips over $100**

The sorted copy is sorted by time, not by amount. A row group is skipped only if its footer proves that no trip in it
costs over $100. The code first shows the `total_amount` statistics of the first 4 row groups in June's Parquet partition, then counts the
summer trips over $100.

**Predict:** does `month_folders_sorted` skip any row group now?

In [ ]:
sm.footer(month_folders_sorted + "pickup_month=2012-06/part-0.parquet",
          column="total_amount", between=(100, 1e9)).head(4)

In [ ]:
over_100_sorted = (spark.read.parquet(month_folders_sorted).where(summer).where("total_amount > 100")
                   .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
over_100_sorted.collect()

In [ ]:
sm.capture("over $100, sorted", over_100_sorted)
sm.checklist("over $100, sorted")

**What to look at.** The 4 row groups shown have a max `total_amount` of $420 to $500, so each could hold a trip
over $100. The checklist: 9 of 9 scan tasks read, all 43,857,527 rows, for 24,279 trips. No row group was skipped:
sorting by time didn't narrow the amounts in each row group.

**Exercise 4. Can the footers answer by themselves?**

A count needs no data: each footer lists its row groups' rows. Spark can add those up without reading the data.

**Predict:** how many rows does the count read? And the latest pickup time, which is in the footers too?

In [ ]:
summer_count = spark.read.parquet(month_folders_sorted).where(summer).selectExpr("count(*)")
summer_count.collect()

In [ ]:
sm.capture("count from footers", summer_count)
sm.checklist("count from footers")

In [ ]:
summer_last_pickup = spark.read.parquet(month_folders_sorted).where(summer).selectExpr("max(tpep_pickup_datetime)")
summer_last_pickup.collect()

In [ ]:
sm.capture("max time from footers?", summer_last_pickup)
sm.checklist("max time from footers?")

**What to look at.** The line `6. read` shows 36 rows read: one row count from the footer for each row group. Spark
adds them up to 43,857,527 trips. The latest pickup time reads all 43,857,527 rows: Spark 3.5 doesn't answer a
timestamp's `max` from the footers. Try `selectExpr("max(total_amount)")` the same way: 36 rows read, or all of them? 

In our tests on Glue, footer answers appeared only with adaptive execution on, which is Glue's default. Open-source
Spark 3.5 doesn't answer from footers by default. Before `spark.read`, turn it on with
`spark.conf.set("spark.sql.parquet.aggregatePushdown", "true")`. Also remove `parquet` from
`spark.sql.sources.useV1SourceList`, for example with
`spark.conf.set("spark.sql.sources.useV1SourceList", "avro,csv,json,kafka,orc,text")`. This makes Spark read Parquet
its newer way.

Even then, Spark answers from footers only `count`, and `min` or `max` of numbers and dates (not of times or text).
The query can't have a condition on a data column, and a group-by, if any, can use only the folder columns.

**Exercise 5. The folders' own rule as a function of the pickup time**

The code counts June's trips with `date_format(tpep_pickup_datetime, 'yyyy-MM') = '2012-06'`, the rule that made
`pickup_month`, without naming `pickup_month`. Spark doesn't know the rule.

**Predict:** does `PartitionFilters` keep 1 folder or all 36?

In [ ]:
june_by_function = (spark.read.parquet(month_folders)
                    .where("date_format(tpep_pickup_datetime, 'yyyy-MM') = '2012-06'")
                    .agg(F.count("*").alias("trips")))
june_by_function.collect()

In [ ]:
sm.capture("june by function", june_by_function)
sm.checklist("june by function")

**What to look at.** `PartitionFilters: (none)`: all 36 folders kept. `PushedFilters` holds only
`IsNotNull(tpep_pickup_datetime)`, so no row group is ruled out either: 72 scan tasks, 520,062,606 rows read for
15,096,468 trips, as in step 3. The condition on `date_format(...)` reaches neither the folder names nor the footers.

**Exercise 6. Folders dropped through a join**

A small table lists four months of 2012 with their seasons; three of them are summer. The query joins on
`pickup_month` and keeps `season = 'summer'`, so it names no month. Spark can still drop folders: it reads the small
table first and turns its summer months into a folder filter while the query runs. This is **dynamic pruning**.

**Predict:** how many rows does it read, and where does the folder filter show up in the plan?

In [ ]:
months_table = spark.read.parquet("s3://436c-2026w1/l7/layouts/months-table/")
summer_by_join = (spark.read.parquet(month_folders)
                  .join(months_table.where("season = 'summer'"), "pickup_month")
                  .agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("total")))
summer_by_join.explain()
summer_by_join.collect()

In [ ]:
sm.capture("summer by join", summer_by_join)
sm.checklist("summer by join")

**What to look at.** In the plan, the taxi scan's `PartitionFilters` holds `dynamicpruningexpression`. The
checklist: 3 reading tasks and 43,857,527 rows read, the trips of the three summer months (the count in exercise 2). The
checklist's 8 files include the months table's own files.

In [ ]:
%stop_session